# Notebook 05 — Feature Engineering

**EvaliSense: AI-Powered Handwritten Examination Evaluation with ML-Based Grading Error Prediction**

---

## Objective

This notebook demonstrates the 16-dimensional feature vector extracted from each evaluation, used as input to the grading error prediction model.

We will:
1. Define and explain all 16 features
2. Extract features from real evaluations (rubric + answer)
3. Analyze the synthetic training data feature distributions
4. Compute class separation (error vs non-error) for each feature
5. Visualize the feature correlation matrix
6. Rank features by discriminative power

### Feature Groups

| Group | Features | Source |
|-------|----------|--------|
| OCR Quality | `ocr_confidence`, `num_lines`, `avg_line_length`, `text_length` | HTR Pipeline |
| Semantic | `evaluation_confidence`, `overall_semantic_similarity`, `rubric_coverage` | Evaluator |
| Score | `normalized_score`, `max_criterion_score`, `min_criterion_score`, `score_variance` | Score Analysis |
| Keywords | `keyword_match_ratio`, `num_criteria` | Rubric Matching |
| Risk Signals | `mark_deviation_from_mean_criterion`, `confidence_score_disagreement`, `score_range` | Derived |

**Why this matters:** Feature engineering determines what signals the ML model can learn from. Good features produce better classifiers with less data.

## Setup

In [ ]:
import os, sys, warnings, json, time
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config import config
from evaluation import Rubric, AnswerEvaluator
from features import extract_features, FEATURE_NAMES

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 120

RESULTS_DIR = config.results_dir
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Feature vector dimensionality: {len(FEATURE_NAMES)}")
print(f"Feature names:")
for i, name in enumerate(FEATURE_NAMES):
    print(f"  {i+1:2d}. {name}")
print("\nSetup complete ✓")

## 1. Extract Features from Real Evaluations

In [ ]:
# Load rubric and evaluator
rubric_path = config.samples_dir / 'rubric_biology_photosynthesis.json'
answers_path = config.samples_dir / 'sample_answers.json'

with open(rubric_path) as f:
    rubric = Rubric.from_dict(json.load(f))
with open(answers_path) as f:
    answers_data = json.load(f)['answers']

evaluator = AnswerEvaluator(model_name='all-MiniLM-L6-v2', similarity_threshold=0.55)

# Extract features for photosynthesis answers
feature_records = []
for a in answers_data:
    if 'photosynthesis' not in a['rubric_file']:
        continue
    
    eval_result = evaluator.evaluate(a['answer_text'], rubric)
    
    mock_htr = {
        'full_text': a['answer_text'],
        'average_confidence': 0.85 if a['quality'] == 'good' else 0.70 if a['quality'] == 'partial' else 0.55,
        'lines': [{'text': line, 'confidence': 0.80, 'line_number': i+1}
                  for i, line in enumerate(a['answer_text'].split('\n')) if line.strip()],
    }
    
    fv = extract_features(mock_htr, eval_result.to_dict())
    rec = fv.to_dict()
    rec['quality'] = a['quality']
    rec['answer_id'] = a['id']
    rec['score'] = eval_result.total_score
    feature_records.append(rec)
    
    print(f"\n{a['id']} ({a['quality']}): score={eval_result.total_score:.1f}/{eval_result.max_score:.0f}")
    print(f"  Feature vector: {np.round(fv.values, 3).tolist()}")

In [ ]:
# Feature comparison by quality
if feature_records:
    viz_features = ['ocr_confidence', 'overall_semantic_similarity', 'evaluation_confidence',
                    'rubric_coverage', 'normalized_score', 'keyword_match_ratio',
                    'score_variance', 'confidence_score_disagreement']
    
    quality_colors = {'good': '#2ECC71', 'partial': '#F39C12', 'poor': '#E74C3C'}
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    fig.suptitle('Feature Values by Answer Quality', fontsize=16, fontweight='bold')
    
    for ax, feat_name in zip(axes.flat, viz_features):
        vals = {q: [] for q in quality_colors}
        for rec in feature_records:
            vals[rec['quality']].append(rec.get(feat_name, 0))
        
        for q in quality_colors:
            if vals[q]:
                ax.bar(q, np.mean(vals[q]), color=quality_colors[q], edgecolor='white', width=0.6)
        ax.set_title(feat_name.replace('_', '\n'), fontsize=9)
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'feature_by_quality.png', dpi=150, bbox_inches='tight')
    plt.show()

## 2. Synthetic Data Feature Analysis

Analyze the 300-record synthetic dataset to understand feature distributions and their relationship to grading errors.

In [ ]:
syn_path = config.samples_dir / 'synthetic_dataset.jsonl'
syn_records = []
with open(syn_path) as f:
    for line in f:
        syn_records.append(json.loads(line.strip()))

errors = [r for r in syn_records if r['is_grading_error']]
non_errors = [r for r in syn_records if not r['is_grading_error']]
feat_keys = sorted(syn_records[0]['features'].keys())

print(f"Records: {len(syn_records)} | Errors: {len(errors)} | Non-errors: {len(non_errors)}")

# Class separation analysis
print(f"\n{'Feature':<42} {'Non-Err Mean':>12} {'Err Mean':>10} {'|Δ|':>8} {'Dir'}")
print("─" * 80)

deltas = {}
for k in feat_keys:
    ne_mean = np.mean([r['features'][k] for r in non_errors])
    e_mean = np.mean([r['features'][k] for r in errors])
    delta = abs(e_mean - ne_mean)
    deltas[k] = delta
    direction = '⬆' if e_mean > ne_mean else '⬇'
    print(f"  {k:<40} {ne_mean:>12.4f} {e_mean:>10.4f} {delta:>7.4f} {direction}")

In [ ]:
# Feature importance ranking by class separation
sorted_features = sorted(deltas.items(), key=lambda x: x[1], reverse=True)

fig, ax = plt.subplots(figsize=(14, 8))
names = [f[0] for f in sorted_features]
values = [f[1] for f in sorted_features]
colors = ['#E74C3C' if v > 0.1 else '#F39C12' if v > 0.05 else '#95A5A6' for v in values]

ax.barh(range(len(names)), values, color=colors, edgecolor='white')
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel('Mean Absolute Difference (Error vs Non-Error)')
ax.set_title('Feature Discriminative Power (Class Separation)', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for i, v in enumerate(values):
    ax.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'feature_importance_separation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR / 'feature_importance_separation.png'}")

In [ ]:
# Correlation matrix
data_matrix = np.array([[r['features'][k] for k in feat_keys] for r in syn_records])
corr = np.corrcoef(data_matrix.T)

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(feat_keys)))
ax.set_yticks(range(len(feat_keys)))
short = [k.replace('_', '\n')[:18] for k in feat_keys]
ax.set_xticklabels(short, fontsize=7, rotation=45, ha='right')
ax.set_yticklabels(short, fontsize=7)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.colorbar(im, label='Pearson Correlation')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR / 'feature_correlation.png'}")

## Observations

1. **Most Discriminative Features:** `confidence_score_disagreement`, `score_variance`, and `normalized_score` show the largest separation between error and non-error classes.

2. **OCR Confidence:** Lower OCR confidence correlates with higher error risk — misread text leads to incorrect scoring.

3. **Feature Correlation:** `normalized_score` and `overall_semantic_similarity` are highly correlated (expected, as both measure answer quality). The ML model must handle this redundancy.

4. **Quality Gradient:** Features show a clear gradient from good → partial → poor answers, confirming the features capture meaningful quality signals.

## Conclusion

The 16-dimensional feature vector captures complementary signals from OCR quality, semantic evaluation, scoring patterns, and keyword coverage. The features show strong class separation, making them suitable for ML classification.

**Next:** [Notebook 06 — ML Model Training](./06_model_training.ipynb)